## RAG pipeline - Data ingestion to vector db pipeline


In [21]:
import os 
from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from pathlib import Path

In [22]:
### Read all pdfs inside the directory

def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")



Found 2 PDF files to process

Processing: Ilya Sutskever - NIPS-2012-imagenet-classification-with-deep-convolutional-neural-networks-Paper.pdf
  ✓ Loaded 9 pages

Processing: Character Win Prediction Analysis.pdf
  ✓ Loaded 11 pages

Total documents loaded: 20


In [23]:
all_pdf_documents

[Document(metadata={'producer': 'Python PDF Library - http://pybrary.net/pyPdf/', 'creator': 'PyPDF', 'creationdate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'created': '2012', 'description-abstract': 'We trained a large, deep convolutional neural network to classify the 1.3 million high-resolution images in the LSVRC-2010 ImageNet training set into the 1000 different classes. On the test data, we achieved top-1 and top-5 error rates of 39.7\\% and 18.9\\% which is considerably better than the previous state-of-the-art results. The neural network, which has 60 million parameters and 500,000 neurons, consists of five convolutional layers, some of which are followed by max-pooling layers, and two globally connected layers with a final 1000-way softmax. To make training faster, we used non-saturating neurons and a very efficient GPU implementation of convolutional nets. To reduce overfitting in th

In [24]:
### Time for chunking - Text Splitter

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, 
    chunk_overlap=chunk_overlap,
    length_function= len,
    separators= ["\n\n", "\n", " ", ""]
    
    )
    split_docs = text_splitter.split_documents(documents)

    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    if split_docs:
        print(f"Example chunk:\n Content:{split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

chunked_documents = split_documents(all_pdf_documents)

# Convert chunks to text
texts = [doc.page_content for doc in chunked_documents]

# Generate embeddings
embeddings = embedding_manager.generate_embeddings(texts)

# Store in vector DB
vectorstore.add_documents(chunked_documents, embeddings)

Split 20 documents into 98 chunks
Example chunk:
 Content:ImageNet Classiﬁcation with Deep Convolutional
Neural Networks
Alex Krizhevsky
University of Toronto
kriz@cs.utoronto.ca
Ilya Sutskever
University of Toronto
ilya@cs.utoronto.ca
Geoffrey E. Hinton
Uni...
Metadata: {'producer': 'Python PDF Library - http://pybrary.net/pyPdf/', 'creator': 'PyPDF', 'creationdate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'created': '2012', 'description-abstract': 'We trained a large, deep convolutional neural network to classify the 1.3 million high-resolution images in the LSVRC-2010 ImageNet training set into the 1000 different classes. On the test data, we achieved top-1 and top-5 error rates of 39.7\\% and 18.9\\% which is considerably better than the previous state-of-the-art results. The neural network, which has 60 million parameters and 500,000 neurons, consists of five convolutional layers, some of 

Batches: 100%|██████████| 4/4 [00:01<00:00,  3.24it/s]

Generated embeddings with shape: (98, 384)
Adding 98 documents to vector store...
Successfully added 98 documents to vector store
Total documents in collection: 196


### Embeddings and vector DB

In [25]:
import numpy as np 
from sentence_transformers import SentenceTransformer
import uuid
import chromadb
from typing import List, Dict, Tuple, Any
from chromadb.config import Settings
from sklearn.metrics.pairwise import cosine_similarity
from dotenv import load_dotenv
load_dotenv()

True

In [26]:
class EmbeddingManager:
    def __init__(self,model_name :str='all-MiniLM-L6-v2'):
        
        self.model_name= model_name
        self.model= None
        self._load_model()
    
    def _load_model(self):
        try:
            self.model= SentenceTransformer(self.model_name)
            print(f"Model '{self.model_name}' loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model '{self.model_name}': {e}")
            raise 
    
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings= self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    

    def get_embedding_dimension(self) -> int:
        if not self.model:
            raise ValueError("Model not loaded")
        return self.model.get_embedding_dimension()


# initialize embedding manager

embedding_manager= EmbeddingManager()
embedding_manager

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7036.62it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model 'all-MiniLM-L6-v2' loaded successfully. Embedding dimension: 384


### Vector Store

In [27]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 196


### Retriever Pipeline from VectorStore

In [31]:
class RAGRetriever:

    def __init__(self, vector_store: vectorstore, embedding_manager:EmbeddingManager):
        
        self.vector_store= vectorstore
        self.embedding_manager= EmbeddingManager()
    
    
    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6273.52it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model 'all-MiniLM-L6-v2' loaded successfully. Embedding dimension: 384


In [29]:
rag_retriever

In [35]:
rag_retriever.retrieve("What is a pokemon")

Retrieving documents for query: 'What is a pokemon'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.35it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


[{'id': 'doc_791fcc60_47',
  'content': 'PokemonTeamPredictionsWhichcharacteristicsof aPokemongivethebest winningrateandcantheybeusedfor prediction?\nSparshKhareDepartment of Computer ScienceFloridaStateUniversityTallahassee, FLUSAsk22bo@fsu.edu\nSabrinaCallejoDepartment of StatisticsFloridaStateUniversityTallahassee, FLUSAsmc19a@fsu.edu\nMauricioEspinozaDepartment of Computer ScienceFloridaStateUniversityTallahassee, FLUSAmespinoza@fsu.edu',
  'metadata': {'source': '../data/pdf_files/Character Win Prediction Analysis.pdf',
   'file_type': 'pdf',
   'page_label': '1',
   'total_pages': 11,
   'page': 0,
   'creator': 'PyPDF',
   'creationdate': '',
   'content_length': 400,
   'source_file': 'Character Win Prediction Analysis.pdf',
   'producer': 'Skia/PDF m114 Google Docs Renderer',
   'title': 'Final Report Draft.docx',
   'doc_index': 47},
  'similarity_score': 0.09837692975997925,
  'distance': 0.9016230702400208,
  'rank': 1},
 {'id': 'doc_1bc95425_47',
  'content': 'PokemonTeamP

### Integration vectordb context pipeline with LLM output


In [43]:
### Simple rag pipeline with groq llm
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

# initialize groq llm
groq_api_key= os.getenv("GROQ_API_KEY")

llm= ChatGroq(groq_api_key= groq_api_key, model_name= "llama-3.1-8b-instant", temperature=0.1, max_tokens=1024)

## create simple RAG function

def rag_simple(query, retriever, llm, top_k=3):
    results= retriever.retrieve(query, top_k)
    context= "\n\n".join(doc['content'] for doc in results) if results else ""
    if not context:
        return "No relevant context form"
    
    # generate answer using groq llm
    prompt= f""" Use the following context to answer the question concisely
        Context: {context} 
        Question: {query}
        Answer:
        """

    response= llm.invoke([prompt.format(context=context, query=query)])
    return response.content


In [47]:
answer= rag_simple("What is xgboost", rag_retriever, llm)
print(answer)

Retrieving documents for query: 'What is xgboost'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.97it/s]


Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)
Extreme Gradient Boosting (XGBoost) is a machine learning algorithm that uses ensemble methods to create an optimal model by iteratively creating individual models, each of which influences the following model. It is built upon the Gradient Boosting algorithm and has gained popularity due to its ability to handle parsed data, regularization, speed, flexibility, and performance.


### Enhanced RAG pipeline features


In [49]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Tell me about SVM", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'Tell me about SVM'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:03<00:00,  3.15s/it]


Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)
Answer: The Support Vector Machine (SVM) algorithm is a supervised learning model that uses hyperparameters to achieve the best performance. It has three key hyperparameters: 

1. **Kernel**: specifies the type of function used to transform data into a higher-dimensional space.
2. **C**: a regularization parameter that controls the trade-off between margin and misclassification error.
3. **Gamma**: a parameter that controls the influence of each data point on the margin.
Sources: [{'source': 'Character Win Prediction Analysis.pdf', 'page': 6, 'score': 0.16643691062927246, 'preview': 'TheSupport Vector Machine(SVM)algorithmalsohashyperparametersthatneedto betunedto achieve thebestperformance.Oneofthemostimportanthyperparametersis thekernel,whichspeciﬁesthetypeoffunctionusedtotransformthedatainto a higher-dimensionalspace.TheotherimportanthyperparametersareCandgamma.Cisthe...'}, {'source': 'Character Win Pr

In [53]:
answer1= llm.invoke("tell me about inland taipan")


In [54]:
answer1.content

"The inland taipan (Oxyuranus microlepidotus) is a venomous snake species native to the deserts and semi-deserts of central Australia. It is considered one of the most venomous snakes in the world, with a potent neurotoxin-based venom that can cause severe systemic symptoms.\n\n**Physical Characteristics:**\n\n- The inland taipan is a medium-sized snake, typically growing to an average length of 1.5-2 meters (4.9-6.6 feet) with a maximum recorded length of 2.5 meters (8.2 feet).\n- It has a slender, elongated body with a brown or grayish-brown color, often with a lighter-colored belly.\n- The snake's head is narrow and pointed, with a distinctive pattern of dark blotches or spots on the back.\n\n**Habitat and Distribution:**\n\n- The inland taipan is found in the arid and semi-arid regions of central Australia, including the Simpson Desert, the Great Victoria Desert, and the Tanami Desert.\n- It inhabits areas with sandy or rocky terrain, often near water sources or in areas with spars

In [56]:
answer1.usage_metadata

{'input_tokens': 42, 'output_tokens': 531, 'total_tokens': 573}